In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

In [2]:
train_df = pd.read_csv("data/train.csv",encoding="utf-8")
test_df = pd.read_csv("data/test.csv",encoding="utf-8")

In [3]:
import os
print(os.getcwd())

C:\Users\Berkay\PycharmProjects\sandbox\titanic_comp


In [4]:
train_df.isna().sum()

PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64

In [5]:
train_df.count()

PassengerId    891
Survived       891
Pclass         891
Name           891
Sex            891
Age            714
SibSp          891
Parch          891
Ticket         891
Fare           891
Cabin          204
Embarked       889
dtype: int64

In [6]:
train_df.head(5)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [7]:
title_map = {
    "Mr": "Mr",
    "Miss": "Miss",
    "Mlle": "Miss",
    "Ms": "Miss",
    "Mrs": "Mrs",
    "Mme": "Mrs",
    "Master": "Master",
    "Rev": "Rare",
    "Dr": "Rare",
    "Col": "Rare",
    "Major": "Rare",
    "Don": "Rare",
    "Dona": "Rare",
    "Lady": "Rare",
    "Sir": "Rare",
    "Capt": "Rare",
    "the Countess": "Rare",
    "Jonkheer": "Rare",
}

In [10]:
def preprocess_df(df):

    drop_cols = ["PassengerId","Name","Ticket",]

    #get title
    get_name = df["Name"].apply(lambda x: x.split(",")[1])
    get_title = get_name.apply(lambda x: x.split(".")[0])
    df["Title"] = get_title
    df["Title"] = df["Title"].str.strip().map(title_map)

    #get Cabin title
    cabin_list = df["Cabin"].astype(str).to_list()
    cabins = []
    for x in cabin_list:
        if str(x) == "nan":
            cabins.append("Unknown")
        else:
            cabins.append(x[0])
    df["Cabin_Title"] = cabins

    #Embarked
    df["Embarked"] = df["Embarked"].fillna("S") ##fill embarked na with S as its the most common one

    #Gender
    df["Sex"] = df["Sex"].map({"male":1, "female":0}) ## map sex col to 0 and 1

    #Ticket
    df['TicketCount'] = df.groupby('Ticket')['Ticket'].transform('count') ##counts how many people holding the same ticket
    df = df.drop(columns='Ticket') ##drops Ticket after

    #add features
    df["Family_size"] = df["SibSp"] + df["Parch"] + 1
    df["isAlone"] = df["Family_size"] == 1
    df["GirlWithParents"] = ((df["Parch"] > 0) & (df["Title"] == "Miss")).astype(int)

    df = pd.get_dummies(df, columns=['Embarked', 'Title'], drop_first=True, dtype=int) ##turn the columns into one hot

    return df


In [11]:
train_pro_df = preprocess_df(train_df)

In [12]:
train_df.groupby(["Sex","Embarked"]).count()

,,PassengerId,Survived,Pclass,Name,Age,SibSp,Parch,Ticket,Fare,Cabin,Title,Cabin_Title,TicketCount
Sex,Embarked,,,,,,,,,,,,,


In [13]:
train_df.iloc[train_df["Embarked"].isna()]

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,Title,Cabin_Title,TicketCount


In [14]:
train_pro_df.isna().sum()

PassengerId          0
Survived             0
Pclass               0
Name                 0
Sex                891
Age                177
SibSp                0
Parch                0
Fare                 0
Cabin              687
Cabin_Title          0
TicketCount          0
Family_size          0
isAlone              0
GirlWithParents      0
Embarked_Q           0
Embarked_S           0
Title_Miss           0
Title_Mr             0
Title_Mrs            0
Title_Rare           0
dtype: int64

In [15]:
res_cabin = train_pro_df.drop(["Cabin","Cabin_Title","Name","PassengerId"], axis=1)

In [16]:
res_cabin.head(5)

,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,TicketCount,Family_size,isAlone,GirlWithParents,Embarked_Q,Embarked_S,Title_Miss,Title_Mr,Title_Mrs,Title_Rare
0,0,3,NaN,22.0,1,0,7.2500,1,2,False,0,0,1,0,1,0,0
1,1,1,NaN,38.0,1,0,71.2833,1,2,False,0,0,0,0,0,1,0
2,1,3,NaN,26.0,0,0,7.9250,1,1,True,0,0,1,1,0,0,0
3,1,1,NaN,35.0,1,0,53.1000,2,2,False,0,0,1,0,0,1,0
4,0,3,NaN,35.0,0,0,8.0500,1,1,True,0,0,1,0,1,0,0


In [17]:
from sklearn.preprocessing import StandardScaler

fare_scaler = StandardScaler()
res_cabin["Fare"] = fare_scaler.fit_transform(res_cabin[["Fare"]])

In [18]:
res_cabin.head(5)

,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,TicketCount,Family_size,isAlone,GirlWithParents,Embarked_Q,Embarked_S,Title_Miss,Title_Mr,Title_Mrs,Title_Rare
0,0,3,NaN,22.0,1,0,-0.502445,1,2,False,0,0,1,0,1,0,0
1,1,1,NaN,38.0,1,0,0.786845,1,2,False,0,0,0,0,0,1,0
2,1,3,NaN,26.0,0,0,-0.488854,1,1,True,0,0,1,1,0,0,0
3,1,1,NaN,35.0,1,0,0.420730,2,2,False,0,0,1,0,0,1,0
4,0,3,NaN,35.0,0,0,-0.486337,1,1,True,0,0,1,0,1,0,0


In [19]:
res_cabin.isna().sum()

Survived             0
Pclass               0
Sex                891
Age                177
SibSp                0
Parch                0
Fare                 0
TicketCount          0
Family_size          0
isAlone              0
GirlWithParents      0
Embarked_Q           0
Embarked_S           0
Title_Miss           0
Title_Mr             0
Title_Mrs            0
Title_Rare           0
dtype: int64

In [20]:
age_known = res_cabin[res_cabin["Age"].notna()]
age_missing = res_cabin[res_cabin["Age"].isna()]
age_X = age_known.drop(columns=["Age", "Survived"])
age_y = age_known["Age"]

In [25]:
from sklearn.model_selection import train_test_split

X_train,X_test,y_train,y_test = train_test_split(age_X, age_y, test_size=0.2, random_state=42)

In [26]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

model = RandomForestRegressor(n_estimators=500, random_state=42)
model.fit(X_train, y_train)

pred = model.predict(X_test)
print("Model MAE:", mean_absolute_error(y_test, pred))

# baseline: just guess the median age every time
print("Baseline MAE:", mean_absolute_error(y_test, [y_train.median()] * len(y_test)))

Model MAE: 8.355314450910603
Baseline MAE: 10.804195804195805


In [28]:
print(X_train.nunique()[X_train.nunique() <= 1])

Sex    0
dtype: int64


In [27]:
from sklearn.ensemble import HistGradientBoostingRegressor
gb = HistGradientBoostingRegressor(random_state=42).fit(X_train, y_train)
print(mean_absolute_error(y_test, gb.predict(X_test)))

ValueError: window shape cannot be larger than input array shape